# 01 — Sentinel-2 Single Dataset Preprocessing Pipeline
## Automated Satellite Retrieval, Cloud Masking & 256x256 GeoTIFF Export

### Overview
This pipeline ingests geographic coordinate bounding boxes, downloads or synthesizes multi-temporal Sentinel-2 imagery, applies `eo-learn` cloud/shadow masking and temporal median compositing, and exports standardized **256x256** GeoTIFFs along with a 3-panel `comparison.jpg` for every coordinate.

**Core Rules Enforced:**
- **Input**: `input/coordinates.csv`
- **Output**: `output/single_dataset/<sample_id>/*.tif` + `comparison.jpg`
- **Fixed Size**: Strictly **256x256** pixels for seamless input into downstream machine learning / super-resolution models.
- **Numbered Band Folders**: `1/`, `2/`, `3/`, `4/` for individual spectral bands (B02, B03, B04, B08).
- **Hard Cap**: `MAX_SAMPLES = 5` for fast testing and iteration.
- **Security**: CDSE credentials loaded only via environment variables (never hardcoded).


## 1. Environment Setup & Dependency Installation
Detects whether running in Google Colab or Local workspace and installs minimal dependencies.


In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("[INFO] Google Colab detected.")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"[WARN] Drive mount skipped: {e}")
else:
    print("[INFO] Local Environment detected.")

# Colab minimal dependency install:
# !pip install -q eo-learn-core==1.5.7 sentinelhub==3.12.0 "setuptools<81" tifffile rasterio shapely pyproj python-dotenv matplotlib pillow


[INFO] Local Environment detected.


## 2. Imports & Configuration
Configures paths to `input/` and `output/`, security parameters, and target size (256, 256).


In [2]:
import csv
import json
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from dotenv import load_dotenv

load_dotenv(override=True)

from single_dataset_pipeline import (
    PipelineConfig,
    SentinelDataset,
    SentinelPreprocessor,
    run_pipeline,
    validate_outputs,
    validate_and_extract_rectangle,
)

config = PipelineConfig(
    data_root=Path("input"),
    output_root=Path("output"),
    dataset_name="single_dataset",
    coords_file="coordinates.csv",
    max_samples=5,
    max_dates=3,
    threads=3,
)

print(f"Input Path  : {config.data_root.resolve()}")
print(f"Output Path : {config.output_root.resolve()}")
print(f"Target Size : {config.target_size} (Strict 256x256)")


C:\Users\advik\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Input Path  : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\input
Output Path : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\output
Target Size : (256, 256) (Strict 256x256)


## 3. Inspect Input Coordinates (`input/coordinates.csv`)
Verifies the input coordinates, terrain types, and geographic bounds.


In [3]:
with open(config.data_root / config.coords_file, "r") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
print(f"Total coordinate targets defined: {len(rows)}")
for r in rows:
    lat, lon, bbox_rect = validate_and_extract_rectangle(r)
    print(f"  ID: {r['sample_id']} | Rect BBox: {[round(x, 4) for x in bbox_rect]} | Window: {r['start_date']} to {r['end_date']}")


Total coordinate targets defined: 5
  ID: sample_01 | Rect BBox: [np.float64(76.9441), 11.0053, np.float64(76.9675), 11.0283] | Window: 2025-11-01 to 2026-02-28
  ID: sample_02 | Rect BBox: [np.float64(76.9008), 10.956, np.float64(76.9242), 10.979] | Window: 2025-11-01 to 2026-02-28
  ID: sample_03 | Rect BBox: [np.float64(77.0093), 10.9907, np.float64(77.0327), 11.0137] | Window: 2025-11-01 to 2026-02-28
  ID: sample_04 | Rect BBox: [np.float64(76.8403), 11.0335, np.float64(76.8637), 11.0565] | Window: 2025-11-01 to 2026-02-28
  ID: sample_05 | Rect BBox: [np.float64(77.1133), 11.0135, np.float64(77.1367), 11.0365] | Window: 2025-11-01 to 2026-02-28


## 4. PyTorch Dataset & DataLoader
Streams samples with `batch_size=1, shuffle=False, num_workers=0`.


In [4]:
dataset = SentinelDataset(config.data_root, coords_file=config.coords_file, dataset_name=config.dataset_name)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda b: b[0])
print(f"Loaded {len(dataset)} validated rectangular targets into PyTorch Dataset.")
sample_0 = dataset[0]
print(f"Sample 0: ID={sample_0['sample_id']}, Centre=({sample_0['latitude']}, {sample_0['longitude']}), BBox={sample_0['bbox_rect']}")


2026-09-28 10:25:24,872 [INFO] Loaded 5 validated rectangular target(s) from input\coordinates.csv


Loaded 5 validated rectangular targets into PyTorch Dataset.
Sample 0: ID=sample_01, Centre=(11.0168, 76.9558), BBox=(np.float64(76.94408573707098), 11.005301616960114, np.float64(76.96751426292901), 11.028298383039886)


## 5. Execute End-to-End Pipeline (Capped at 5 Samples)
Processes each coordinate: SCL cloud masking, median compositing, 256x256 GeoTIFF export, and `comparison.jpg` creation.


In [5]:
results = run_pipeline(config)


2026-09-28 10:25:24,879 [INFO] ======================================================================


2026-09-28 10:25:24,880 [INFO] STARTING SINGLE DATASET PREPROCESSING PIPELINE (LIVE SATELLITE MODE)


2026-09-28 10:25:24,881 [INFO] Input Directory  : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\input


2026-09-28 10:25:24,884 [INFO] Output Directory : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\output


2026-09-28 10:25:24,886 [INFO] Output Size      : (256, 256) (Strict 256x256)


...

  3 dates, 256x256 px -> downloading with 3 threads


2026-09-28 10:26:04,438 [INFO]   [OK] 'sample_05' -> 256x256 GeoTIFFs & comparison.jpg verified on disk.


2026-09-28 10:26:04,439 [INFO] ======================================================================


2026-09-28 10:26:04,439 [INFO] PIPELINE COMPLETE: 5/5 Succeeded | Output: output\single_dataset


2026-09-28 10:26:04,440 [INFO] ======================================================================


## 6. Visual Verification (`comparison.jpg` for Every Coordinate)
Displays the 3-panel before/mask/after preview for each coordinate.


In [6]:
for r in results:
    if r["status"] == "SUCCESS":
        jpg_path = r["result"]["comparison_path"]
        print(f"=== Visual Comparison: {r['sample_id']} ===")
        img = Image.open(jpg_path)
        plt.figure(figsize=(15, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Coordinate {r['sample_id']} Comparison Preview", fontsize=12, fontweight="bold")
        plt.show()


=== Visual Comparison: sample_01 ===
=== Visual Comparison: sample_02 ===
=== Visual Comparison: sample_03 ===
=== Visual Comparison: sample_04 ===
=== Visual Comparison: sample_05 ===


C:\Users\advik\AppData\Local\Temp\ipykernel_12496\3075007310.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Output Validation & 256x256 Size Confirmation
Confirms all exported files exist on disk, have valid byte sizes, and are strictly 256x256 pixels.


In [7]:
out_dir = config.output_root / config.dataset_name
sample_folders = sorted([d for d in out_dir.iterdir() if d.is_dir()])
print("=" * 80)
print(f"{'Sample ID':<15} {'Composite TIFF':<25} {'Shape':<18} {'Comparison File':<20} {'Status'}")
print("=" * 80)
for sf in sample_folders:
    tif_f = sf / "composite_reflectance.tif"
    jpg_f = sf / "comparison.jpg"
    arr = tifffile.imread(str(tif_f))
    shape_str = str(arr.shape)
    is_256 = arr.shape[-2:] == (256, 256)
    print(f"{sf.name:<15} {tif_f.name:<25} {shape_str:<18} {jpg_f.name:<20} {'256x256 VALID' if is_256 else 'INVALID'}")
print("=" * 80)


Sample ID       Composite TIFF            Shape              Comparison File      Status
sample_01       composite_reflectance.tif (4, 256, 256)      comparison.jpg       256x256 VALID
sample_02       composite_reflectance.tif (4, 256, 256)      comparison.jpg       256x256 VALID
sample_03       composite_reflectance.tif (4, 256, 256)      comparison.jpg       256x256 VALID
sample_04       composite_reflectance.tif (4, 256, 256)      comparison.jpg       256x256 VALID
sample_05       composite_reflectance.tif (4, 256, 256)      comparison.jpg       256x256 VALID
